<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/S%C3%BAly%20Regulariz%C3%A1ci%C3%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Súly Regularizáció

Ebben a notebookban a **súly regularizáció** módszereit vizsgáljuk.

## Tartalomjegyzék

1. Regularizáció motiváció
2. L2 regularizáció (Ridge)
3. L1 regularizáció (Lasso)
4. Elastic Net
5. Weight decay vs L2

## 1. Regularizáció motiváció

### Overfitting probléma

| Modell | Tréning hiba | Teszt hiba |
|--------|--------------|------------|
| Underfitting | Nagy | Nagy |
| Jó | Közepes | Közepes |
| Overfitting | Kicsi | Nagy |

### Regularizáció hatása

$$L_{reg}(\theta) = L(\theta) + \lambda R(\theta)$$

- $L(\theta)$: eredeti veszteség
- $R(\theta)$: regularizációs tag
- $\lambda$: regularizáció erőssége

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet

np.random.seed(42)
torch.manual_seed(42)

# Adat generálás
def generate_data(n_samples=100, noise=0.5):
    X = np.linspace(0, 1, n_samples).reshape(-1, 1)
    y = np.sin(2 * np.pi * X).flatten() + noise * np.random.randn(n_samples)
    return X, y

X, y = generate_data(n_samples=30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Polinom feature-ök
degree = 15
poly = PolynomialFeatures(degree)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Skálázás
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

# Overfitting demonstráció
from sklearn.linear_model import LinearRegression

# Regularizáció nélkül
model_no_reg = LinearRegression()
model_no_reg.fit(X_train_scaled, y_train)

# Predikció
X_plot = np.linspace(0, 1, 200).reshape(-1, 1)
X_plot_poly = poly.transform(X_plot)
X_plot_scaled = scaler.transform(X_plot_poly)

y_pred_no_reg = model_no_reg.predict(X_plot_scaled)

# Vizualizáció
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, c='blue', s=50, label='Tréning')
plt.scatter(X_test, y_test, c='red', s=50, label='Teszt')
plt.plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', linewidth=2, label='Valódi függvény')
plt.plot(X_plot, y_pred_no_reg, 'r-', linewidth=2, label='Regularizáció nélkül')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Overfitting: {degree}. fokú polinom regularizáció nélkül')
plt.legend()
plt.ylim(-3, 3)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Train MSE: {np.mean((model_no_reg.predict(X_train_scaled) - y_train)**2):.4f}")
print(f"Test MSE: {np.mean((model_no_reg.predict(X_test_scaled) - y_test)**2):.4f}")

## 2. L2 regularizáció (Ridge)

### Definíció

$$L_{L2} = L + \lambda \sum_i w_i^2 = L + \lambda ||\mathbf{w}||_2^2$$

### Gradiens

$$\frac{\partial L_{L2}}{\partial w_i} = \frac{\partial L}{\partial w_i} + 2\lambda w_i$$

### Hatás

- Kis súlyok felé tol
- Nem zérusít ki súlyokat
- Simább modell

In [ ]:
# L2 regularizáció különböző erősségekkel

alphas = [0, 0.001, 0.01, 0.1, 1, 10]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

train_errors = []
test_errors = []
coefficients = []

for ax, alpha in zip(axes, alphas):
    if alpha == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=alpha)

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_plot_scaled)

    train_mse = np.mean((model.predict(X_train_scaled) - y_train)**2)
    test_mse = np.mean((model.predict(X_test_scaled) - y_test)**2)

    train_errors.append(train_mse)
    test_errors.append(test_mse)
    coefficients.append(model.coef_)

    ax.scatter(X_train, y_train, c='blue', s=30)
    ax.scatter(X_test, y_test, c='red', s=30)
    ax.plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', linewidth=2)
    ax.plot(X_plot, y_pred, 'r-', linewidth=2)
    ax.set_title(f'λ={alpha}\nTrain: {train_mse:.3f}, Test: {test_mse:.3f}')
    ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle('L2 (Ridge) regularizáció', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Súlyok alakulása

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train vs Test error
axes[0].plot(alphas, train_errors, 'bo-', linewidth=2, markersize=10, label='Train')
axes[0].plot(alphas, test_errors, 'ro-', linewidth=2, markersize=10, label='Test')
axes[0].set_xlabel('λ (regularizáció erőssége)')
axes[0].set_ylabel('MSE')
axes[0].set_title('Train vs Test hiba')
axes[0].set_xscale('symlog', linthresh=0.001)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Súlyok
for i, (alpha, coef) in enumerate(zip(alphas, coefficients)):
    axes[1].plot(range(len(coef)), coef, 'o-', label=f'λ={alpha}', alpha=0.7)
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('Súly értéke')
axes[1].set_title('Súlyok alakulása L2 regularizációval')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. L1 regularizáció (Lasso)

### Definíció

$$L_{L1} = L + \lambda \sum_i |w_i| = L + \lambda ||\mathbf{w}||_1$$

### Szubgradiens

$$\frac{\partial L_{L1}}{\partial w_i} = \frac{\partial L}{\partial w_i} + \lambda \cdot \text{sign}(w_i)$$

### Hatás

- Sparse (ritka) megoldások
- Feature selection
- Súlyokat nullázhat

In [ ]:
# L1 vs L2 összehasonlítás

alpha = 0.1

model_l1 = Lasso(alpha=alpha, max_iter=10000)
model_l2 = Ridge(alpha=alpha)

model_l1.fit(X_train_scaled, y_train)
model_l2.fit(X_train_scaled, y_train)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Predikciók
y_pred_l1 = model_l1.predict(X_plot_scaled)
y_pred_l2 = model_l2.predict(X_plot_scaled)

axes[0].scatter(X_train, y_train, c='blue', s=50, label='Tréning')
axes[0].plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', linewidth=2, label='Valódi')
axes[0].plot(X_plot, y_pred_l1, 'r-', linewidth=2, label='L1 (Lasso)')
axes[0].plot(X_plot, y_pred_l2, 'b-', linewidth=2, label='L2 (Ridge)')
axes[0].set_title(f'Predikciók (λ={alpha})')
axes[0].set_ylim(-2, 2)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Súlyok összehasonlítása
x_pos = np.arange(len(model_l1.coef_))
width = 0.35

axes[1].bar(x_pos - width/2, model_l1.coef_, width, label='L1 (Lasso)', color='red', alpha=0.7)
axes[1].bar(x_pos + width/2, model_l2.coef_, width, label='L2 (Ridge)', color='blue', alpha=0.7)
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('Súly')
axes[1].set_title('Súlyok összehasonlítása')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Nemnulla súlyok száma
l1_nonzero = np.sum(np.abs(model_l1.coef_) > 1e-10)
l2_nonzero = np.sum(np.abs(model_l2.coef_) > 1e-10)

axes[2].bar(['L1 (Lasso)', 'L2 (Ridge)'], [l1_nonzero, l2_nonzero],
            color=['red', 'blue'], alpha=0.7)
axes[2].set_ylabel('Nemnulla súlyok száma')
axes[2].set_title(f'Sparsity (összes: {len(model_l1.coef_)})')

plt.tight_layout()
plt.show()

print(f"L1 nemnulla súlyok: {l1_nonzero}/{len(model_l1.coef_)}")
print(f"L2 nemnulla súlyok: {l2_nonzero}/{len(model_l2.coef_)}")

## 4. Elastic Net

### Definíció

$$L_{EN} = L + \lambda_1 ||\mathbf{w}||_1 + \lambda_2 ||\mathbf{w}||_2^2$$

vagy sklearn paraméterezéssel:

$$L_{EN} = L + \alpha \cdot \rho ||\mathbf{w}||_1 + \alpha \cdot \frac{1-\rho}{2} ||\mathbf{w}||_2^2$$

ahol $\rho$ (l1_ratio) a két regularizáció aránya.

### Előnyök

- L1 és L2 előnyei kombinálva
- Csoportos feature selection
- Korrelált feature-ök kezelése

In [ ]:
# Elastic Net különböző l1_ratio értékekkel

l1_ratios = [0.0, 0.25, 0.5, 0.75, 1.0]
alpha = 0.1

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for ax, l1_ratio in zip(axes[:-1], l1_ratios):
    if l1_ratio == 0:
        model = Ridge(alpha=alpha)
        name = 'Ridge (L2)'
    elif l1_ratio == 1:
        model = Lasso(alpha=alpha, max_iter=10000)
        name = 'Lasso (L1)'
    else:
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
        name = f'ElasticNet ({l1_ratio})'

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_plot_scaled)

    test_mse = np.mean((model.predict(X_test_scaled) - y_test)**2)
    n_nonzero = np.sum(np.abs(model.coef_) > 1e-10)

    ax.scatter(X_train, y_train, c='blue', s=30)
    ax.plot(X_plot, np.sin(2 * np.pi * X_plot), 'g--', linewidth=2)
    ax.plot(X_plot, y_pred, 'r-', linewidth=2)
    ax.set_title(f'{name}\nTest MSE: {test_mse:.3f}, Nemnulla: {n_nonzero}')
    ax.set_ylim(-2, 2)
    ax.grid(True, alpha=0.3)

# L1/L2 constraint szemléltetés
ax = axes[-1]
theta = np.linspace(0, 2*np.pi, 100)

# L2 (kör)
ax.plot(np.cos(theta), np.sin(theta), 'b-', linewidth=2, label='L2 (kör)')

# L1 (rombusz)
l1_x = [1, 0, -1, 0, 1]
l1_y = [0, 1, 0, -1, 0]
ax.plot(l1_x, l1_y, 'r-', linewidth=2, label='L1 (rombusz)')

ax.set_aspect('equal')
ax.set_title('L1 vs L2 constraint geometria')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Elastic Net: L1 és L2 kombinációja', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Weight decay vs L2

### Különbség adaptív optimalizálóknál

| Módszer | L2 | Weight Decay |
|---------|----|--------------|
| SGD | Ekvivalens | Ekvivalens |
| Adam | Skálázott | Valódi decay |

### Adam + L2

$$g_t = \nabla L + \lambda \theta$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

L2 tag az adaptív lr-rel együtt skálázódik!

### AdamW (Weight Decay)

$$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t \right)$$

Weight decay független az adaptív lr-től!

In [ ]:
# PyTorch weight decay példa

from sklearn.datasets import make_classification

X_cls, y_cls = make_classification(n_samples=500, n_features=50,
                                   n_informative=10, n_redundant=20,
                                   random_state=42)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

scaler_cls = StandardScaler()
X_train_cls = scaler_cls.fit_transform(X_train_cls)
X_test_cls = scaler_cls.transform(X_test_cls)

X_train_t = torch.FloatTensor(X_train_cls)
y_train_t = torch.FloatTensor(y_train_cls)
X_test_t = torch.FloatTensor(X_test_cls)
y_test_t = torch.FloatTensor(y_test_cls)

def train_model(optimizer_class, weight_decay, epochs=100):
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(50, 100),
        nn.ReLU(),
        nn.Linear(100, 50),
        nn.ReLU(),
        nn.Linear(50, 1),
        nn.Sigmoid()
    )

    optimizer = optimizer_class(model.parameters(), lr=0.01, weight_decay=weight_decay)
    criterion = nn.BCELoss()

    train_losses = []
    test_losses = []

    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # Test
        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t).squeeze()
            test_loss = criterion(test_out, y_test_t)
            test_losses.append(test_loss.item())

    # Súlyok L2 normája
    weight_norm = sum(p.data.norm(2).item()**2 for p in model.parameters())**0.5

    return train_losses, test_losses, weight_norm

# Összehasonlítás
weight_decays = [0, 0.001, 0.01, 0.1]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for wd in weight_decays:
    train_l, test_l, wn = train_model(optim.Adam, wd)
    axes[0, 0].plot(train_l, label=f'wd={wd}')
    axes[0, 1].plot(test_l, label=f'wd={wd}')

axes[0, 0].set_title('Train Loss (Adam)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_title('Test Loss (Adam)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# AdamW vs Adam összehasonlítás
wd = 0.01
train_adam, test_adam, _ = train_model(optim.Adam, wd)
train_adamw, test_adamw, _ = train_model(optim.AdamW, wd)

axes[1, 0].plot(train_adam, 'b-', linewidth=2, label='Adam + L2')
axes[1, 0].plot(train_adamw, 'r-', linewidth=2, label='AdamW')
axes[1, 0].set_title(f'Train Loss (weight_decay={wd})')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(test_adam, 'b-', linewidth=2, label='Adam + L2')
axes[1, 1].plot(test_adamw, 'r-', linewidth=2, label='AdamW')
axes[1, 1].set_title(f'Test Loss (weight_decay={wd})')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Weight Decay hatása', fontsize=14)
plt.tight_layout()
plt.show()

## Összefoglalás

### Regularizációs módszerek:

| Módszer | Képlet | Hatás |
|---------|--------|-------|
| L2 | $\lambda \|\mathbf{w}\|_2^2$ | Kis súlyok |
| L1 | $\lambda \|\mathbf{w}\|_1$ | Sparse súlyok |
| Elastic | L1 + L2 | Kombinált |

### Mikor melyiket:

| Szituáció | Ajánlott |
|-----------|----------|
| Feature selection | L1 |
| Általános reg. | L2 |
| Korrelált feat. | Elastic Net |

### Adaptív optimalizálók:

- Adam + weight_decay ≠ Adam + L2
- AdamW preferált neurális hálókhoz